# 03 — Add Graph Features (A4)
SIH26182 — Duo A (Data & ML)

**Goal:** Add centrality/graph-based features (in-degree, out-degree, PageRank), retrain, and show the
before/after improvement. This is your strongest talking point — log both numbers side by side.


In [8]:
import pandas as pd
import networkx as nx
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

merged_df = pd.read_csv("merged_data.csv")
edges_df = pd.read_csv("data/elliptic_txs_edgelist.csv")
print(merged_df.shape, edges_df.shape)
edges_df.head()


(46564, 168) (234355, 2)


,txId1,txId2
0,230425980,5530458
1,232022460,232438397
2,230460314,230459870
3,230333930,230595899
4,232013274,232029206


## Build the directed transaction graph

In [9]:
G = nx.from_pandas_edgelist(
    edges_df,
    source=edges_df.columns[0],
    target=edges_df.columns[1],
    create_using=nx.DiGraph(),
)
print("Nodes:", G.number_of_nodes(), " Edges:", G.number_of_edges())


Nodes: 203769  Edges: 234355


## Compute in-degree, out-degree, PageRank for every node

In [10]:
in_deg = dict(G.in_degree())
out_deg = dict(G.out_degree())
pagerank = nx.pagerank(G)

graph_feats = pd.DataFrame({
    "txId": list(G.nodes()),
    "in_degree": [in_deg.get(n, 0) for n in G.nodes()],
    "out_degree": [out_deg.get(n, 0) for n in G.nodes()],
    "pagerank": [pagerank.get(n, 0.0) for n in G.nodes()],
})
graph_feats.head()


,txId,in_degree,out_degree,pagerank
0,230425980,1,1,0.000005
1,5530458,1,1,0.000006
2,232022460,1,2,0.000007
3,232438397,160,1,0.000351
4,230460314,2,8,0.000002


## Merge graph features into the training table (fill missing with 0)

In [11]:
merged_v2 = merged_df.merge(graph_feats, on="txId", how="left")
merged_v2[["in_degree", "out_degree", "pagerank"]] = merged_v2[["in_degree", "out_degree", "pagerank"]].fillna(0)

print("merged_v2 shape:", merged_v2.shape)
merged_v2.to_csv("merged_data_v2.csv", index=False)
merged_v2.head()


merged_v2 shape: (46564, 171)


,txId,time_step,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,...,feat_160,feat_161,feat_162,feat_163,feat_164,feat_165,label,in_degree,out_degree,pagerank
0,232438397,1,0.163054,1.963790,-0.646376,12.409294,-0.063725,9.782742,12.414558,-0.163645,...,1.072793,0.085530,-0.131155,0.677799,-0.120613,-0.119792,licit,160,1,0.000351
1,232029206,1,-0.005027,0.578941,-0.091383,4.380281,-0.063725,4.667146,0.851305,-0.163645,...,0.604120,0.008632,-0.131155,0.333211,-0.120613,-0.119792,licit,59,1,0.000085
2,232344069,1,-0.147852,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.137933,...,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792,licit,0,2,0.000002
3,27553029,1,-0.151357,-0.184668,-1.201369,-0.121970,-0.043875,-0.113002,-0.061584,-0.141519,...,0.018279,-0.087490,-0.131155,-0.097524,-0.120613,-0.119792,licit,1,1,0.000003
4,3881097,1,-0.172306,-0.184668,-1.201369,0.028105,-0.043875,-0.029140,0.242712,-0.163640,...,0.018279,-0.068266,-0.084674,-0.054450,-1.760926,-1.760984,licit,1,1,0.000002


## Retrain on the expanded feature set and compare to baseline
Uses the **same temporal split** as `02_train_baseline.ipynb` (train on time_step 1-34, test on
35-49) so the before/after comparison is apples-to-apples — a different split would make the
"improvement" meaningless.

In [12]:
baseline_feature_columns = [c for c in merged_df.columns if c.startswith("feat_") or c == "time_step"]
graph_feature_columns = baseline_feature_columns + ["in_degree", "out_degree", "pagerank"]

train_mask = merged_v2["time_step"] <= 34
test_mask = merged_v2["time_step"] > 34

X_train = merged_v2.loc[train_mask, graph_feature_columns]
y_train = merged_v2.loc[train_mask, "label"]
X_test = merged_v2.loc[test_mask, graph_feature_columns]
y_test = merged_v2.loc[test_mask, "label"]

clf_v2 = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
clf_v2.fit(X_train, y_train)
y_pred_v2 = clf_v2.predict(X_test)

precision_v2 = precision_score(y_test, y_pred_v2, pos_label="illicit")
recall_v2 = recall_score(y_test, y_pred_v2, pos_label="illicit")
f1_v2 = f1_score(y_test, y_pred_v2, pos_label="illicit")

print("=== BASELINE (from 02_train_baseline.ipynb) vs WITH GRAPH FEATURES ===")
print(f"{'Metric':<12}{'Baseline':<12}{'+Graph Feats':<12}")
# Paste your baseline numbers from notebook 02 here for the live side-by-side, e.g.:
baseline_precision, baseline_recall, baseline_f1 = 0.916, 0.725, 0.809  # actual numbers from 02_train_baseline.ipynb
print(f"{'Precision':<12}{baseline_precision:<12.3f}{precision_v2:<12.3f}")
print(f"{'Recall':<12}{baseline_recall:<12.3f}{recall_v2:<12.3f}")
print(f"{'F1-score':<12}{baseline_f1:<12.3f}{f1_v2:<12.3f}")
print()
print("Confusion matrix (with graph features), labels=[illicit, licit]:")
print(confusion_matrix(y_test, y_pred_v2, labels=["illicit", "licit"]))


=== BASELINE (from 02_train_baseline.ipynb) vs WITH GRAPH FEATURES ===
Metric      Baseline    +Graph Feats
Precision   0.916       0.902       
Recall      0.725       0.725       
F1-score    0.809       0.804       

Confusion matrix (with graph features), labels=[illicit, licit]:
[[  785   298]
 [   85 15502]]


✅ **Verify (A4):** A clear before/after comparison table — even a small F1 improvement is a legitimate, presentable
result. **The comparison itself is the talking point**, not the raw score. Don't skip logging both numbers side by side —
this is your strongest slide.

In [13]:
joblib.dump(clf_v2, "model_v2.pkl")
print("Saved model_v2.pkl")


Saved model_v2.pkl
